In [ ]:
!pip install sentence-transformers

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from lightgbm import LGBMRegressor
from sentence_transformers import SentenceTransformer

In [ ]:
train_df = pd.read_csv("/content/train.csv")

train_df.shape

In [ ]:
X = train_df["catalog_content"]
y = train_df["price"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_valid.shape)

In [ ]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

In [ ]:
X_train_embeddings = model.encode(
    X_train.tolist(),
    show_progress_bar=True
)

X_valid_embeddings = model.encode(
    X_valid.tolist(),
    show_progress_bar=True
)

In [ ]:
print(X_train_embeddings.shape)
print(X_valid_embeddings.shape)

In [ ]:
lgbm_model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

lgbm_model.fit(
    X_train_embeddings,
    y_train
)

In [ ]:
predictions = lgbm_model.predict(
    X_valid_embeddings
)

In [ ]:
mae = mean_absolute_error(
    y_valid,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_valid,
        predictions
    )
)

r2 = r2_score(
    y_valid,
    predictions
)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

In [ ]:
import re

def extract_quantity_features(text):
    text = str(text).lower()

    ounce = re.search(r'(\d+\.?\d*)\s*(oz|ounce)', text)
    pound = re.search(r'(\d+\.?\d*)\s*(lb|pound)', text)
    pack = re.search(r'pack of (\d+)', text)
    serving = re.search(r'(\d+)\s*servings', text)
    count = re.search(r'(\d+)\s*count', text)

    return pd.Series([
        float(ounce.group(1)) if ounce else 0,
        float(pound.group(1)) if pound else 0,
        float(pack.group(1)) if pack else 0,
        float(serving.group(1)) if serving else 0,
        float(count.group(1)) if count else 0
    ])


quantity_features = train_df[
    "catalog_content"
].apply(extract_quantity_features)

quantity_features.columns = [
    "ounce_feature",
    "pound_feature",
    "pack_feature",
    "serving_feature",
    "count_feature"
]

quantity_features.head()

In [ ]:
from scipy.sparse import hstack
import numpy as np

quantity_train = quantity_features.loc[
    X_train.index
].values

quantity_valid = quantity_features.loc[
    X_valid.index
].values

X_train_final = np.hstack([
    X_train_embeddings,
    quantity_train
])

X_valid_final = np.hstack([
    X_valid_embeddings,
    quantity_valid
])

print(X_train_final.shape)
print(X_valid_final.shape)

In [ ]:
lgbm_hybrid = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

lgbm_hybrid.fit(
    X_train_final,
    y_train
)

In [ ]:
hybrid_predictions = lgbm_hybrid.predict(
    X_valid_final
)

In [ ]:
mae = mean_absolute_error(
    y_valid,
    hybrid_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_valid,
        hybrid_predictions
    )
)

r2 = r2_score(
    y_valid,
    hybrid_predictions
)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)